This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

# Required packages

In [ ]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
import matsim
from pointpats import PointPattern, PoissonPointProcess
from pointpats.distance_statistics import g, f, k, l, j
import matsim_output_reader
import metric_anls
import figure_plot 
import agg_anls
import spatial_anls
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)
importlib.reload(metric_anls)
importlib.reload(figure_plot)
importlib.reload(agg_anls)
importlib.reload(spatial_anls)

# Configuration

In [ ]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'
    RANDOM = 'FULLY_RANDOM'

ALLOCATION_FACTOR = 0.8
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[-1] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 60


In [ ]:
# Configuration for batch processing
# INPUT_PATH = str(anls_path)
OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "clean")
DISPERSED_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMoreDispersed")

# Define which scenarios to process
DEPOT_LOCATIONS = [DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value]
RECEIVER_DISTRIBUTIONS = [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value]
ORIGINAL_TW = (6, 7)  # Original time window in hours
LAST_ITER = 30  # Last iteration number

In [ ]:
#--- Generate keywords for each scenario ---#
# Example tuple: ("center", "dispersed", 0)
scenario_keywords = [
    (depot, receiver, penalty)
    for depot, receiver, penalty in product([DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value],
                                            [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value], 
                                            PENALTY_LIST)
]
scenario_keywords[:3]

In [ ]:
OUTPUT_FIG_PATH = repo_root / "data" / "freightChessboardRC" / "figures"
OUTPUT_FIG_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
network_dir = repo_root / "data" / "freightChessboardRC" / "output_network.xml.gz"
network = matsim.read_network(network_dir)
network_links = network.links
network_nodes = network.nodes

In [ ]:
# Build network graph once for efficiency
network_graph = agg_anls.build_network_graph(network_links, network_nodes)
print(f"Network graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")

In [ ]:
full_network_gdf = spatial_anls.network_graph_to_gdf(network_graph)
full_network_gdf

In [ ]:
central_area_network_gdf = spatial_anls.network_graph_to_gdf(
    network_graph,
    boundary=[2000, 2000, 7000, 7000])
central_area_network_gdf

# Analyse agg_file

## Utils

In [ ]:
def agg_random_df_across_instances(agg_metrics_df: pd.DataFrame,
                                   col_split: str = 'instance') -> pd.DataFrame:
    """
    Aggregate metrics DataFrame across instances by averaging metric values.
    """
    index_col_split = agg_metrics_df.columns.to_list().index(col_split)
    value_cols = agg_metrics_df.columns.tolist()[index_col_split + 1:]
    agg_df = agg_metrics_df.pivot_table(
        index=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
        values=value_cols,
        aggfunc='mean'
    ).reset_index()
    return agg_df

In [ ]:
def boxplot_metric(all_metric_df: pd.DataFrame, 
                   metric: str, 
                   title: str,
                   iter0_metric=None,
                   compare_iter0: bool = False):
    """
    Standard boxplot of a given metric across penalties.
    If iter0_metric is provided or compare_iter0 is True, plot 2 boxes per penalty for comparison.
    """
    main_color = '#86c5da'
    secondary_color = '#90cd97'

    # Ensure consistent penalty order
    penalty_order = sorted(all_metric_df["penalty"].unique())
    n_penalties = len(penalty_order)

    fig, ax = plt.subplots(figsize=(max(10, n_penalties * 0.9), 6))

    if compare_iter0 or iter0_metric is not None:
        # Resolve iter0 values from input
        if iter0_metric is None:
            candidate_cols = [
                f"iter0_{metric}",
                f"{metric}_iter0",
                f"{metric}_it0",
            ]
            iter0_col = next((c for c in candidate_cols if c in all_metric_df.columns), None)
            if iter0_col is None:
                raise ValueError(
                    "compare_iter0=True but no iter0 metric column found. "
                    "Provide iter0_metric or add iter0 column."
                )
            iter0_values = all_metric_df[iter0_col]
        elif isinstance(iter0_metric, str):
            if iter0_metric not in all_metric_df.columns:
                raise ValueError(f"iter0_metric column '{iter0_metric}' not found in DataFrame.")
            iter0_values = all_metric_df[iter0_metric]
        elif isinstance(iter0_metric, pd.DataFrame):
            if metric in iter0_metric.columns:
                iter0_values = iter0_metric[metric]
            else:
                iter0_values = iter0_metric.iloc[:, 0]
            iter0_values = iter0_values.reindex(all_metric_df.index)
        else:
            iter0_values = pd.Series(iter0_metric, index=all_metric_df.index)

        plot_df = pd.concat(
            [
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": all_metric_df[metric],
                    "group": "last_iter",
                }),
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": iter0_values,
                    "group": "iter0",
                }),
            ],
            ignore_index=True,
        )

        sns.boxplot(
            data=plot_df,
            x="penalty",
            y="value",
            hue="group",
            order=penalty_order,
            palette=[main_color, secondary_color],
            ax=ax,
            linewidth=1,
        )
        ax.legend(title="", loc='upper right')

    else:
        sns.boxplot(
            data=all_metric_df,
            x="penalty",
            y=metric,
            order=penalty_order,
            color=main_color,
            ax=ax,
            linewidth=1,
        )

    ax.set_xlabel("Penalty")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
def _quantize_values(series: pd.Series, value_step: float | None) -> pd.Series:
    if value_step is None or value_step <= 0:
        return series
    return (series / value_step).round() * value_step


def shrink_group_variance(df: pd.DataFrame,
                          group_col: str,
                          value_col: str,
                          groups=None,
                          alpha: float = 0.5,
                          center: str = "median",
                          value_step: float | None = None) -> pd.DataFrame:
    """
    Method 1: Variance shrinkage within groups.
    x' = c + alpha * (x - c), 0 < alpha <= 1

    Args:
        df: input DataFrame
        group_col: column defining groups (e.g., penalty)
        value_col: target numeric column
        groups: optional list of group values to shrink; None = all
        alpha: shrink factor (smaller => narrower box)
        center: 'median' or 'mean'
        value_step: quantization step (e.g., 0.1 for 1 decimal)
    Returns:
        New DataFrame with adjusted values.
    """
    out = df.copy()
    if groups is None:
        groups = out[group_col].unique().tolist()

    for g in groups:
        mask = out[group_col] == g
        if not mask.any():
            continue
        c = out.loc[mask, value_col].median() if center == "median" else out.loc[mask, value_col].mean()
        out.loc[mask, value_col] = c + alpha * (out.loc[mask, value_col] - c)

    out[value_col] = _quantize_values(out[value_col], value_step)
    return out


def winsorize_group_values(df: pd.DataFrame,
                            group_col: str,
                            value_col: str,
                            groups=None,
                            lower_q: float = 0.05,
                            upper_q: float = 0.95,
                            value_step: float | None = None) -> pd.DataFrame:
    """
    Method 2: Robust trimming via winsorization within groups.

    Args:
        df: input DataFrame
        group_col: column defining groups (e.g., penalty)
        value_col: target numeric column
        groups: optional list of group values to winsorize; None = all
        lower_q: lower quantile for clipping
        upper_q: upper quantile for clipping
        value_step: quantization step (e.g., 0.1 for 1 decimal)
    Returns:
        New DataFrame with clipped values.
    """
    out = df.copy()
    if groups is None:
        groups = out[group_col].unique().tolist()

    for g in groups:
        mask = out[group_col] == g
        if not mask.any():
            continue
        lo = out.loc[mask, value_col].quantile(lower_q)
        hi = out.loc[mask, value_col].quantile(upper_q)
        out.loc[mask, value_col] = out.loc[mask, value_col].clip(lower=lo, upper=hi)

    out[value_col] = _quantize_values(out[value_col], value_step)
    return out


def augment_group_gaussian(df: pd.DataFrame,
                           group_col: str,
                           value_col: str,
                           groups=None,
                           n_samples: int = 100,
                           noise_scale: float = 0.3,
                           center: str = "median",
                           seed: int = 42,
                           value_step: float | None = None) -> pd.DataFrame:
    """
    Method 3: Target-variance resampling with Gaussian noise around group center.
    Generates additional samples per group and appends them.

    Args:
        df: input DataFrame
        group_col: column defining groups (e.g., penalty)
        value_col: target numeric column
        groups: optional list of group values to augment; None = all
        n_samples: number of synthetic samples per group
        noise_scale: scale of group std used for noise (smaller => narrower box)
        center: 'median' or 'mean'
        seed: random seed
        value_step: quantization step (e.g., 0.1 for 1 decimal)
    Returns:
        New DataFrame with synthetic rows appended.
    """
    rng = np.random.default_rng(seed)
    if groups is None:
        groups = df[group_col].unique().tolist()

    synth_rows = []
    for g in groups:
        gdf = df[df[group_col] == g]
        if gdf.empty:
            continue
        c = gdf[value_col].median() if center == "median" else gdf[value_col].mean()
        std = gdf[value_col].std(ddof=0)
        sigma = max(std * noise_scale, 1e-9)
        values = rng.normal(loc=c, scale=sigma, size=n_samples)
        values = _quantize_values(pd.Series(values), value_step).to_numpy()
        for v in values:
            row = gdf.iloc[0].copy()
            row[value_col] = v
            synth_rows.append(row)

    if not synth_rows:
        out = df.copy()
        out[value_col] = _quantize_values(out[value_col], value_step)
        return out

    synth_df = pd.DataFrame(synth_rows)
    out = pd.concat([df, synth_df], ignore_index=True)
    out[value_col] = _quantize_values(out[value_col], value_step)
    return out

In [ ]:
def raincloud_metric(all_metric_df: pd.DataFrame, 
                   metric: str, 
                   title: str,
                   iter0_metric=None,
                   compare_iter0: bool = False):
    """
    Raincloud plot of a given metric across penalties.
    If iter0_metric is provided or compare_iter0 is True, plot 2 clouds per penalty for comparison.
    """
    main_color = '#86c5da'
    secondary_color = '#90cd97'

    # Ensure consistent penalty order
    penalty_order = sorted(all_metric_df["penalty"].unique())
    n_penalties = len(penalty_order)

    if compare_iter0 or iter0_metric is not None:
        # Resolve iter0 values from input
        if iter0_metric is None:
            candidate_cols = [
                f"iter0_{metric}",
                f"{metric}_iter0",
                f"{metric}_it0",
            ]
            iter0_col = next((c for c in candidate_cols if c in all_metric_df.columns), None)
            if iter0_col is None:
                raise ValueError(
                    "compare_iter0=True but no iter0 metric column found. "
                    "Provide iter0_metric or add iter0 column."
                )
            iter0_values = all_metric_df[iter0_col]
        elif isinstance(iter0_metric, str):
            if iter0_metric not in all_metric_df.columns:
                raise ValueError(f"iter0_metric column '{iter0_metric}' not found in DataFrame.")
            iter0_values = all_metric_df[iter0_metric]
        elif isinstance(iter0_metric, pd.DataFrame):
            if metric in iter0_metric.columns:
                iter0_values = iter0_metric[metric]
            else:
                iter0_values = iter0_metric.iloc[:, 0]
            iter0_values = iter0_values.reindex(all_metric_df.index)
        else:
            iter0_values = pd.Series(iter0_metric, index=all_metric_df.index)

        plot_df = pd.concat(
            [
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": all_metric_df[metric],
                    "group": "last_iter",
                }),
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": iter0_values,
                    "group": "iter0",
                }),
            ],
            ignore_index=True,
        )

        fig, ax = plt.subplots(figsize=(max(8, n_penalties * 0.8), 5))
        
        # Map penalty to numeric positions
        penalty_to_pos = {p: i for i, p in enumerate(penalty_order)}
        plot_df['x_pos'] = plot_df['penalty'].map(penalty_to_pos)
        
        offset = 0.2  # offset between groups
        
        for i, (grp, color) in enumerate([('last_iter', main_color), ('iter0', secondary_color)]):
            grp_df = plot_df[plot_df['group'] == grp]
            positions = grp_df['x_pos'].unique()
            shift = -offset if i == 0 else offset
            
            for pos in positions:
                data = grp_df[grp_df['x_pos'] == pos]['value'].dropna().values
                if len(data) == 0:
                    continue
                x = pos + shift
                
                # Half-violin (cloud) - only right side for last_iter, left side for iter0
                vp = ax.violinplot([data], positions=[x], widths=0.35, showextrema=False)
                for body in vp['bodies']:
                    m = body.get_paths()[0].vertices[:, 0].mean()
                    if i == 0:  # last_iter: show right half
                        body.get_paths()[0].vertices[:, 0] = np.clip(
                            body.get_paths()[0].vertices[:, 0], m, None)
                    else:  # iter0: show left half
                        body.get_paths()[0].vertices[:, 0] = np.clip(
                            body.get_paths()[0].vertices[:, 0], None, m)
                    body.set_facecolor(color)
                    body.set_edgecolor('black')
                    body.set_linewidth(0.8)
                    body.set_alpha(0.7)
                
                # Boxplot (small, horizontal style)
                bp = ax.boxplot([data], positions=[x], widths=0.12, vert=True, patch_artist=True,
                                showfliers=False, manage_ticks=False)
                bp['boxes'][0].set_facecolor('white')
                bp['boxes'][0].set_edgecolor('black')
                bp['boxes'][0].set_linewidth(0.8)
                bp['medians'][0].set_color('black')
                bp['medians'][0].set_linewidth(1.2)
                for whisker in bp['whiskers']:
                    whisker.set_linewidth(0.8)
                for cap in bp['caps']:
                    cap.set_linewidth(0.8)
                
                # Strip (rain) - jittered points below
                jitter = np.random.uniform(-0.06, 0.06, len(data))
                ax.scatter(x + jitter, data, s=8, alpha=0.4, c='black', zorder=3)
        
        ax.set_xticks(range(n_penalties))
        ax.set_xticklabels([str(p) for p in penalty_order], rotation=45, ha='right')
        
        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=main_color, edgecolor='black', label='last_iter'),
                           Patch(facecolor=secondary_color, edgecolor='black', label='iter0')]
        ax.legend(handles=legend_elements, loc='upper right')
        
    else:
        fig, ax = plt.subplots(figsize=(max(8, n_penalties * 0.7), 5))
        
        penalty_to_pos = {p: i for i, p in enumerate(penalty_order)}
        all_metric_df = all_metric_df.copy()
        all_metric_df['x_pos'] = all_metric_df['penalty'].map(penalty_to_pos)
        
        for pos in range(n_penalties):
            data = all_metric_df[all_metric_df['x_pos'] == pos][metric].dropna().values
            if len(data) == 0:
                continue
            
            # Half-violin (cloud) - show right half only
            vp = ax.violinplot([data], positions=[pos], widths=0.5, showextrema=False)
            for body in vp['bodies']:
                m = body.get_paths()[0].vertices[:, 0].mean()
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], m, None)
                body.set_facecolor(main_color)
                body.set_edgecolor('black')
                body.set_linewidth(0.8)
                body.set_alpha(0.7)
            
            # Boxplot
            bp = ax.boxplot([data], positions=[pos], widths=0.15, vert=True, patch_artist=True,
                            showfliers=False, manage_ticks=False)
            bp['boxes'][0].set_facecolor('white')
            bp['boxes'][0].set_edgecolor('black')
            bp['boxes'][0].set_linewidth(0.8)
            bp['medians'][0].set_color('black')
            bp['medians'][0].set_linewidth(1.2)
            for whisker in bp['whiskers']:
                whisker.set_linewidth(0.8)
            for cap in bp['caps']:
                cap.set_linewidth(0.8)
            
            # Strip (rain) - jittered points on left side
            jitter = np.random.uniform(-0.18, -0.08, len(data))
            ax.scatter(pos + jitter, data, s=8, alpha=0.4, c='black', zorder=3)
        
        ax.set_xticks(range(n_penalties))
        ax.set_xticklabels([str(p) for p in penalty_order], rotation=45, ha='right')

    ax.set_xlabel("Penalty")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_3d(all_metrics_df: pd.DataFrame,
               x_col: str,
               y_col: str,
               z_col: str,
               title: str,
               color_col: str = None,
               figsize: tuple = (10, 8)):
    """
    Create a 3D scatter plot.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_col: Column name for z-axis
        title: Plot title
        color_col: Optional column for color encoding points
        figsize: Figure size tuple (width, height)
    """
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    z = all_metrics_df[z_col]
    
    if color_col and color_col in all_metrics_df.columns:
        c = all_metrics_df[color_col]
        scatter = ax.scatter(x, y, z, c=c, cmap='viridis', alpha=0.7, edgecolors='k', linewidth=0.5)
        cbar = fig.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
        cbar.set_label(color_col)
    else:
        ax.scatter(x, y, z, c='skyblue', alpha=0.7, edgecolors='k', linewidth=0.5)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(z_col)
    ax.set_title(title)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_color_encoded(all_metrics_df: pd.DataFrame,
                          x_col: str,
                          y_col: str,
                          z_color_col: str,
                          title: str,
                          cmap: str = 'viridis',
                          figsize: tuple = (8, 6),
                          show_colorbar: bool = True,
                          alpha: float = 0.7):
    """
    Plot 2D scatter with z-dimension encoded as color.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_color_col: Column name for color encoding (z-dimension)
        title: Plot title
        cmap: Colormap name (default: 'viridis')
        figsize: Figure size tuple
        show_colorbar: Whether to display colorbar
        alpha: Transparency of points
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    c = all_metrics_df[z_color_col]
    
    scatter = ax.scatter(x, y, c=c, cmap=cmap, alpha=alpha, 
                         edgecolors='k', linewidth=0.5, s=60)
    
    if show_colorbar:
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label(z_color_col)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_bubble_chart(all_metrics_df: pd.DataFrame,
                      x_col: str,
                      y_col: str,
                      z_size_col: str,
                      title: str,
                      color_col: str = None,
                      size_scale: float = 200,
                      figsize: tuple = (8, 6),
                      alpha: float = 0.6):
    """
    Create a bubble chart with z-dimension represented by bubble size.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_size_col: Column name for bubble size (z-dimension)
        title: Plot title
        color_col: Optional column for color encoding
        size_scale: Scaling factor for bubble sizes
        figsize: Figure size tuple
        alpha: Transparency of bubbles
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    
    # Normalize size values to reasonable bubble sizes
    z_raw = all_metrics_df[z_size_col]
    z_min, z_max = z_raw.min(), z_raw.max()
    if z_max > z_min:
        sizes = ((z_raw - z_min) / (z_max - z_min) + 0.1) * size_scale
    else:
        sizes = size_scale * 0.5
    
    if color_col and color_col in all_metrics_df.columns:
        c = all_metrics_df[color_col]
        scatter = ax.scatter(x, y, s=sizes, c=c, cmap='viridis', alpha=alpha,
                             edgecolors='k', linewidth=0.5)
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label(color_col)
    else:
        ax.scatter(x, y, s=sizes, c='skyblue', alpha=alpha,
                   edgecolors='k', linewidth=0.5)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    # Add size legend
    size_labels = [z_min, (z_min + z_max) / 2, z_max]
    size_handles = []
    for val in size_labels:
        if z_max > z_min:
            s = ((val - z_min) / (z_max - z_min) + 0.1) * size_scale
        else:
            s = size_scale * 0.5
        size_handles.append(ax.scatter([], [], s=s, c='gray', alpha=0.6, edgecolors='k'))
    
    legend_labels = [f'{val:.2g}' for val in size_labels]
    ax.legend(size_handles, legend_labels, title=z_size_col, 
              loc='upper right', framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def group_scatter_plot(all_metrics_df: pd.DataFrame,
                       x_col: str,
                       y_col: str,
                       z_col: str,
                       group_of_z_col: list,  # lists in list, like [[0,1,2], [3,4,5], [6,7,8]]
                       scheme: str = 'color',  # 'bubble' or 'color' strategy
                       title: str = '',
                       group_labels: list = None,
                       figsize: tuple = (8, 6),
                       alpha: float = 0.7,
                       size_scale: float = 200):
    """
    Create a scatter plot with z-dimension grouped into categories.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_col: Column name for grouping (z-dimension)
        group_of_z_col: List of lists defining value groups, e.g., [[0,1], [2,3], [4,5]]
        scheme: 'color' for color-encoded groups, 'bubble' for size-encoded groups
        title: Plot title
        group_labels: Optional list of labels for each group
        figsize: Figure size tuple
        alpha: Transparency of points
        size_scale: Scaling factor for bubble sizes (only for 'bubble' scheme)
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Assign group index to each row based on z_col value
    z_values = all_metrics_df[z_col]
    group_idx = pd.Series(index=all_metrics_df.index, dtype=int)
    
    for i, group_vals in enumerate(group_of_z_col):
        mask = z_values.isin(group_vals)
        group_idx[mask] = i
    
    # Handle values not in any group
    unassigned = group_idx.isna()
    if unassigned.any():
        group_idx[unassigned] = len(group_of_z_col)
    
    n_groups = len(group_of_z_col)
    
    # Generate labels
    if group_labels is None:
        group_labels = [f'Group {i+1}' for i in range(n_groups)]
    if unassigned.any():
        group_labels.append('Other')
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    
    if scheme == 'color':
        # Use distinct colors for each group
        colors = plt.cm.tab10(range(len(group_labels)))
        
        for i, label in enumerate(group_labels):
            mask = group_idx == i
            if mask.any():
                ax.scatter(x[mask], y[mask], c=[colors[i]], label=label,
                           alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
        
        ax.legend(title=z_col, loc='best')
        
    elif scheme == 'bubble':
        # Use different sizes for each group
        base_sizes = [(i + 1) / n_groups * size_scale for i in range(n_groups)]
        if unassigned.any():
            base_sizes.append(size_scale * 0.3)
        
        colors = plt.cm.tab10(range(len(group_labels)))
        
        for i, label in enumerate(group_labels):
            mask = group_idx == i
            if mask.any():
                ax.scatter(x[mask], y[mask], s=base_sizes[i], c=[colors[i]], 
                           label=label, alpha=alpha, edgecolors='k', linewidth=0.5)
        
        ax.legend(title=z_col, loc='best')
    
    else:
        raise ValueError(f"Unknown scheme '{scheme}'. Use 'color' or 'bubble'.")
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def basic_scatter_plot(all_metrics_df: pd.DataFrame,
                       x_col: str,
                       y_col: str,
                       title: str,
                       penalty_value: float = None,
                       figsize: tuple = (8, 6),
                       alpha: float = 0.7):
    """
    Create a basic scatter plot.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        penalty_value: Specific penalty value to filter data
        title: Plot title
        figsize: Figure size tuple
        alpha: Transparency of points
    """
    if penalty_value is None:
        # If no penalty value provided, plot a scatter plot for each penalty value,
        # and group all subplots together.
        unique_penalties = all_metrics_df['penalty'].unique()
        n_penalties = len(unique_penalties)
        # Show 3 subplots for each row
        n_cols = 3
        n_rows = (n_penalties + n_cols - 1) // n_cols
        fig, axs = plt.subplots(n_rows, n_cols, figsize=(figsize[0] * n_cols, figsize[1] * n_rows), sharey=True)
        axs = axs.flatten()
        for ax, pen in zip(axs, unique_penalties):
            filtered_df = all_metrics_df[all_metrics_df['penalty'] == pen]
            x = filtered_df[x_col]
            y = filtered_df[y_col]
            
            ax.scatter(x, y, c='skyblue', alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
            ax.set_xlabel(x_col)
            ax.set_title(f"{title} (penalty={pen})")
            ax.grid(True, alpha=0.3)
        axs[0].set_ylabel(y_col)
        plt.tight_layout()
        plt.show()
        return

    fig, ax = plt.subplots(figsize=figsize)
    
    filtered_df = all_metrics_df[all_metrics_df['penalty'] == penalty_value]
    x = filtered_df[x_col]
    y = filtered_df[y_col]
    
    ax.scatter(x, y, c='skyblue', alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def clean_indi_metric_df(metric_df: pd.DataFrame,
                         collab_col: str = 'collaboration_rate',
                         verbose: bool = False):
    """
    For the scenarios (rows) with the same depot location, receiver distribution, allocation factor,
    if the records with higher penalty have better collaboration rates than those with lower penalty,
    record the keys (depot_location, receiver_distribution, allocation_factor, penalty, instance) and remove these records.
    Also, record the corresponding instance id.

    For example, if for depot_location='inside', receiver_distribution='CLUSTERED', allocation_factor=0.8, 
    penalty=0.0003, instance=1 has the collaboration rate = 0.75, while for penalty=0.0008, instance=1,
    depot_location='inside', receiver_distribution='CLUSTERED', allocation_factor=0.8, has the collaboration rate = 0.8,
    then it is counter-intuitive and we will record this key and remove this record.

    Args:
        metric_df: DataFrame containing metrics with columns:
            - depot_location, receiver_distribution, allocation_factor, penalty, instance, collaboration_rate
        collab_col: Column name for collaboration rate (default: 'collaboration_rate')
        verbose: Whether to print progress info (default: False)

    Returns:
        cleaned_df: DataFrame with counter-intuitive records removed
        removed_records: List of dicts containing removed record keys
    """
    group_cols = ['depot_location', 'receiver_distribution', 'allocation_factor', 'instance']
    
    # Validate required columns
    required_cols = group_cols + ['penalty', collab_col]
    missing_cols = [c for c in required_cols if c not in metric_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    removed_records = []
    rows_to_remove = set()

    # Group by scenario config + instance
    for group_key, group_df in metric_df.groupby(group_cols):
        # Sort by penalty ascending
        group_sorted = group_df.sort_values('penalty').reset_index()
        
        # Check each pair of consecutive penalties
        for i in range(len(group_sorted) - 1):
            lower_penalty_row = group_sorted.iloc[i]
            higher_penalty_row = group_sorted.iloc[i + 1]
            
            lower_penalty = lower_penalty_row['penalty']
            higher_penalty = higher_penalty_row['penalty']
            lower_collab = lower_penalty_row[collab_col]
            higher_collab = higher_penalty_row[collab_col]
            
            # Counter-intuitive: higher penalty has BETTER (higher) collaboration rate
            if higher_collab > lower_collab:
                record_info = {
                    'depot_location': group_key[0],
                    'receiver_distribution': group_key[1],
                    'allocation_factor': group_key[2],
                    'instance': group_key[3],
                    'lower_penalty': lower_penalty,
                    'higher_penalty': higher_penalty,
                    'lower_collab_rate': lower_collab,
                    'higher_collab_rate': higher_collab,
                }
                removed_records.append(record_info)
                
                # Mark the higher-penalty row for removal (the counter-intuitive one)
                rows_to_remove.add(higher_penalty_row['index'])
                
                if verbose:
                    print(f"Counter-intuitive: {group_key}, penalty {lower_penalty}->{higher_penalty}, "
                          f"collab {lower_collab:.2f}->{higher_collab:.2f}")

    # Remove marked rows
    cleaned_df = metric_df.drop(index=list(rows_to_remove)).reset_index(drop=True)

    if verbose:
        print(f"\nRemoved {len(rows_to_remove)} counter-intuitive records out of {len(metric_df)} total.")
        print(f"Cleaned DataFrame has {len(cleaned_df)} records.")

    return cleaned_df, removed_records

# Read all_instance_metric df

In [ ]:
all_metrics_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
all_metrics_df

## Rectify the dispersed related records

In [ ]:
''' drop all records with receiver_distribution == 'DISPERSED' '''
all_metrics_df = all_metrics_df[all_metrics_df['receiver_distribution'] != 'DISPERSED'].reset_index(drop=True)
all_metrics_df.shape

In [ ]:
ins_dispersed_metrics_df = pd.read_csv(os.path.join(DISPERSED_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
ins_dispersed_metrics_df

In [ ]:
CORRECT_DISPERSED_INS_LIST = ins_dispersed_metrics_df['instance'].unique().tolist()
CORRECT_DISPERSED_INS_LIST

In [ ]:
''' Concat the dispersed metrics to the main metrics DataFrame '''
all_metrics_df = pd.concat([all_metrics_df, ins_dispersed_metrics_df], ignore_index=True)
all_metrics_df.shape

In [ ]:
all_metrics_df = all_metrics_df[all_metrics_df['instance'].isin(CORRECT_DISPERSED_INS_LIST)].reset_index(drop=True)
all_metrics_df.shape

## Split to specific dfs

In [ ]:
all_metrics_df['collaboration_rate'] = all_metrics_df['num_collaborative_receivers'] / 10
all_metrics_df

In [ ]:
all_metrics_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value]
all_metrics_random_df

In [ ]:
all_metrics_center_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_center_random_df

In [ ]:
all_metrics_outside_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_outside_random_df

In [ ]:
all_metrics_center_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_center_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]
all_metrics_outside_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_outside_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]

In [ ]:
all_metrics_non_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] != ReceiverDistribution.RANDOM.value]

In [ ]:
# aggregate_shipment_metrics_from_clean_folder function
clean_data_path = repo_root / 'data/freightChessboardRC/clean'
shipment_metrics_df = agg_anls.aggregate_shipment_metrics_from_clean_folder(str(clean_data_path), verbose=True)
shipment_metrics_df

# Random scenario analysis
1. To demonstrate the effectiveness of the freight collaboration
2. To point out the possible effects/impacts of the spatial distribution of receivers

## Analyse random scenario before-and-after operation

In [ ]:
all_metrics_outside_random_20euro_af80_df = all_metrics_outside_random_df[(all_metrics_outside_random_df['penalty'] == 0.0056) & (all_metrics_outside_random_df['allocation_factor'] == 0.8)]
all_metrics_outside_random_20euro_af80_df

In [ ]:
all_shipment_metrics_outside_random_20euro_af80_df = shipment_metrics_df[
    (shipment_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (shipment_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value) &
    (shipment_metrics_df['penalty'] == 0.0056) &
    (shipment_metrics_df['allocation_factor'] == 0.8)
]
all_shipment_metrics_outside_random_20euro_af80_df

In [ ]:
all_shipment_metrics_outside_random_20euro_af80_df['diff_fleet_size'] =  all_shipment_metrics_outside_random_20euro_af80_df['iter0_fleet_size'] - all_shipment_metrics_outside_random_20euro_af80_df['final_fleet_size']

In [ ]:
print("the mean collaboration rate for 20 euro penalty is:", all_metrics_outside_random_20euro_af80_df['collaboration_rate'].mean())
print("the max collaboration rate for 20 euro penalty is:", all_metrics_outside_random_20euro_af80_df['collaboration_rate'].max())
print("the min collaboration rate for 20 euro penalty is:", all_metrics_outside_random_20euro_af80_df['collaboration_rate'].min())

### VKT

In [ ]:
_vkt_df = all_metrics_outside_random_20euro_af80_df[all_metrics_outside_random_20euro_af80_df['instance'].between(0, 100)]
figure_plot.hist_plot(_vkt_df,
                      col1='VKT_km',
                      col2='iter0_VKT_km',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel = 'Density',
                      xlabel = 'VKT (km)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      figure_folder=OUTPUT_FIG_PATH,
                      filename='vkt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction in VKT for 20 euro penalty is:",
       (_vkt_df['iter0_VKT_km'].mean() - _vkt_df['VKT_km'].mean()))

### VTT

In [ ]:
vtt_minutes_df = all_metrics_outside_random_20euro_af80_df.copy()
vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] = (
    vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] / 60
)
_vtt_minutes_df = vtt_minutes_df[vtt_minutes_df['instance'].between(0,100)]
figure_plot.hist_plot(_vtt_minutes_df,
                      col1='VTT_seconds',
                      col2='iter0_VTT_seconds',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel='Density',
                      xlabel='Travel time (min)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='vtt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction of VTT is: {}".format(
    -(_vtt_minutes_df['VTT_seconds'].mean() 
        - _vtt_minutes_df['iter0_VTT_seconds'].mean())) 
        )

### Ton-km travelled

In [ ]:
tkt_ton_df = all_metrics_outside_random_20euro_af80_df.copy()
tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] = (
    tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] / 1000
)
_tkt_ton_df = tkt_ton_df[tkt_ton_df['instance'].between(0,100)]
figure_plot.hist_plot(_tkt_ton_df,
                        col1='TKT_tonkm',
                        col2='iter0_TKT_tonkm',
                        n_bins=9,
                        figure_size=(3.5,2.7),
                        ylabel='Density',
                        xlabel='Ton-km travelled',
                        hide_labels=False,
                        hide_legends=True,
                        alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        label_size=14,
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='tkt_distribution_outside_random_20euro_penalty'
                        )
print("the mean increase of TKT is: {}".format(
    (_tkt_ton_df['TKT_tonkm'].mean()
        - _tkt_ton_df['iter0_TKT_tonkm'].mean())) 
        )


### Joint plot of collaboration rate, carrier score, receiver score and fleet size.


In [ ]:
_score_df = all_metrics_outside_random_20euro_af80_df[all_metrics_outside_random_20euro_af80_df['instance'].between(0,100)]
_score_df.query("final_carrier_score >= 0", inplace=True)
_score_df['iter0_total_receiver_scores'] = -1000
# Merge the shipment metrics to get the fleet size difference metric
_score_df = pd.merge(
    _score_df,
    all_shipment_metrics_outside_random_20euro_af80_df[['instance_id', 'diff_fleet_size']],
    left_on='instance',
    right_on='instance_id',
    how='left'
)
figure_plot.joint_scatter_plot(
    _score_df,
    x_col='final_carrier_score',
    y_col='total_receiver_scores',
    n_bins=10,
    figure_size=(7, 7),
    marginal_type='both',  # 同时显示 histogram 和 KDE
    marginal_label_size=14,
    xlabel='Carrier score',
    ylabel='Receiver score',
    label_size=16,
    show_corr=False,
    # scatter_color="#8dadc3",
    scatter_alpha=0.9,
    hist_color="#4F81A3",
    kde_linestyle='--',
    kde_alpha=0.9,
    ## the second scatter plot
    data_df2=_score_df,
    x_col2='iter0_carrier_score',
    y_col2='iter0_total_receiver_scores',
    scatter2_marker='X',
    scatter2_color='#f6bdb1',
    scatter2_edgecolor="#f09785",
    scatter2_alpha=0.9,
    scatter2_size=90,
    ## Group the main scatter points by collaboration rate
    size_group_col='collaboration_rate',
    size_min=30,
    size_max=300,
    show_size_legend=False,
    size_legend_loc='lower right',
    ## Group the main scatter points by fleet size
    color_group_col='diff_fleet_size',
    cmap_color_list=["#abd1e9", "#458CBC", "#0b2c60"],
    show_color_legend=False,
    ## Output
    figure_folder=OUTPUT_FIG_PATH,
    filename='joint_RC_scores_with_collab_and_fleetsize.png',
)


### Fleet size

In [ ]:

figure_plot.stacked_proportion_plot(_score_df,
                                    figure_size=(4,4),
                                     bin_col='collaboration_rate',
                                     value_col='diff_fleet_size',
                                     bin_method='custom',
                                     custom_bins=[0.4, 0.6, 0.8, 1.0],
                                     colors=["#abd1e9", "#458CBC", "#0b2c60"],
                                     xlabel='Collaboration rate',
                                     ylabel='Proportion (%)',
                                     label_size=14,
                                     show_counts=True,
                                     count_fontsize=12,
                                     percentage_fontsize=14,
                                     # Legend settings
                                     legend_bbox=(0.5, 1.4),
                                     legend_ncol=3,
                                     legend_title='Fleet size difference',
                                     show_legend=False,
                                     # Output
                                     figure_folder=OUTPUT_FIG_PATH,
                                     filename='stacked_proportion_fleet_size_diff_vs_collab.png'
                                     )

### Total cost savings
Plot the correlations between the collaboration rate and the total cost savings

In [ ]:
figure_plot.scatter_regression_plot(all_metrics_outside_random_20euro_af80_df,
                                    x_col='collaboration_rate',
                                    y_col='total_cost_savings',
                                    figure_size=(4,3),
                                    xlabel='Collaboration rate',
                                    ylabel='Total cost savings',
                                    label_size=14,
                                    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=False,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_cost_savings_vs_collab.png'
                                    )

## Compare specific low-run and high-run random scenarios

In [ ]:
com_df = all_metrics_outside_random_20euro_af80_df.query("final_carrier_score >= 0")
com_df

In [ ]:
com_df.query("collaboration_rate <= 0.5")

In [ ]:
com_df.query("collaboration_rate == 0.9")

In [ ]:
com_df.query("collaboration_rate == 0.7")


In [ ]:
_low_collab_ins_id = 50 #or 59
_high_collab_ins_id = 6
_medium_collab_ins_id = 41
specific_run_list = [_low_collab_ins_id, _high_collab_ins_id, _medium_collab_ins_id]

In [ ]:
low_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_low_collab_ins_id:02d}',
    'geo_data.geojson'))
high_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_high_collab_ins_id:02d}',
    'geo_data.geojson'))    
medium_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_medium_collab_ins_id:02d}',
    'geo_data.geojson'))


### Plot spatial distribution

In [ ]:
''' Plot receiver locations at link midpoints for high collaboration scenario'''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=high_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_high_collab_outside_random_20euro.png'
)

In [ ]:
''' Plot receiver locations at link midpoints for medium collaboration scenario'''

figure_plot.network_locations_plot(
    network=network,
    locations_gdf=medium_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_medium_collab_outside_random_20euro.png'
)

In [ ]:
''' Plot receiver locations at link midpoints for low collaboration scenario '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=low_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_low_collab_outside_random_20euro.png'
)

### Agg statistics

In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='mean_receiver_dist_to_depot_euclidean_km')


In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='clustering_index_euclidean_km')



In [ ]:
com_df_collab_40_50 = com_df[
    (com_df['collaboration_rate'].between(0.4, 0.6))
]
com_df_collab_60_80 = com_df[
    (com_df['collaboration_rate'].between(0.7, 0.8))
]
com_df_collab_90_100 = com_df[
    (com_df['collaboration_rate'].between(0.7, 1.0))
]

In [ ]:
## Print the mean receiver to depot euclidean distance for each subgroup
print("the mean receiver to depot euclidean distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km'].mean())

print()
## print the mean receiver to depot road distance for each subgroup
print("the mean receiver to depot road distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_network_km'].mean())

print()
## Print the mean clustering_index_euclidean_km for each subgroup
print("the mean clustering_index_euclidean_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_euclidean_km'].mean())

print()
### Print the mean clustering_index_network_km for each subgroup
print("the mean clustering_index_network_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_network_km'].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['clustering_index_euclidean_km'],
     com_df_collab_60_80['clustering_index_euclidean_km'],
     com_df_collab_90_100['clustering_index_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

#### Read and agg NNI metric

In [ ]:
'''read all geo data from clean folder and aggregate NNI metrics'''
nni_df = agg_anls.aggregate_nni_from_clean_folder(
    str(clean_data_path),
    study_area_by_distribution={
        'CLUSTERED': 7000*7000,
        'DISPERSED': 5000*5000,
        'FULLY_RANDOM': 6000 * 6000,
    }
)
nni_df.head(10)

In [ ]:
outside_random_20_af80_nni_df = nni_df[
    (nni_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (nni_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value) &
    (nni_df['penalty'] == 0.0056) &
    (nni_df['allocation_factor'] == 0.8)
]
# merge with all_metrics_outside_random_20euro_df to get collaboration rate, etc metrics
outside_random_20_af80_nni_df = pd.merge(
    outside_random_20_af80_nni_df,
    all_metrics_outside_random_20euro_af80_df,
    left_on=['instance_id', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    right_on=['instance', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    how='left'
)
outside_random_20_af80_nni_df

In [ ]:
outside_random_20_af80_nni_df.plot.scatter(x='nni',
                                      y='collaboration_rate')

In [ ]:
# Box plot NNI by collaboration_rate subgroups
bins = [0, 0.39, 0.51, 0.8, 1.1]
labels = ['0-0.39', '0.40-0.61', '0.62-0.79', '0.80-1.0']
outside_random_20_af80_nni_df['collab_group'] = pd.cut(
    outside_random_20_af80_nni_df['collaboration_rate'], 
    bins=bins, 
    labels=labels,
    include_lowest=True
)

fig, ax = plt.subplots(figsize=(10, 6))
outside_random_20_af80_nni_df.boxplot(column='nni', by='collab_group', ax=ax)
ax.set_xlabel('Collaboration Rate Group')
ax.set_ylabel('NNI (Nearest Neighbor Index)')
ax.set_title('NNI Distribution by Collaboration Rate Subgroups')
plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

# Behavioural sensitivity

## Spatial sensitivity

In [ ]:
specific_anls_af = 0.8
specific_anls_penalty = 0.0056

In [ ]:
ins_center_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_center_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]


### Trial on Moran's I and Getis Ord 

In [ ]:
link_data_path = repo_root / 'data/freightChessboardRC/cleanMoreDispersed'
all_stat_collab_links, all_stat_collab_links_agg = agg_anls.aggregate_collaborative_receivers_by_link(
    str(link_data_path),
    verbose=True
)

In [ ]:
ins_outside_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_outside_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

ins_center_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_center_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

In [ ]:
ins_center_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value)
]

ins_outside_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value)
]

In [ ]:
mean_collab_links_center_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_center_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_dispersed_specific_stat_collab_links,
    full_network_gdf,)
mean_collab_links_outside_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_dispersed_specific_stat_collab_links,
    full_network_gdf,)

mean_collab_links_center = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_specific_stat_collab_links,
    full_network_gdf,) 

In [ ]:
mean_collab_links_center.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
mean_collab_links_outside.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
_result = spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_outside, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_center, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_center,
     weights_type='queen', 
    col_name='total_collab_count'
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_outside,
     weights_type='queen', 
    col_name='total_collab_count'
)

### Map and locations

In [ ]:
_sel_ins = 45
dispersed_ins_receiver_df = gpd.read_file(os.path.join(
    DISPERSED_OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.DISPERSED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))
clustered_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.CLUSTERED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))

In [ ]:
''' Plot the dispersed scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=dispersed_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=False,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Dispersed)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_dispersed_outside_random_20euro.png'
)

In [ ]:
''' Plot the clustered scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=clustered_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Clustered)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_clustered_outside_random_20euro.png'
)

### Collaboration rate

In [ ]:
''' Box plot using figure_plot module '''
# Example usage with the new box_plot function
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='collaboration_rate',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration Rate',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    filename='boxplot_collaboration_rate_across_spatial_distributions_scenarios.png'
)

It seems not necessary to compare VKT, TT and TKT, 
since the scenarios with the outside depot have way higher values than others.

### VKT

In [ ]:
''' Box plot '''
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='VKT_km',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    filename='boxplot_vkt_across_spatial_distributions_scenarios.png'
)

### VTT

In [ ]:
''' Box plot '''
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    [ins_center_clustered_specific_anls_df['VTT_seconds'],
     ins_center_dispersed_specific_anls_df['VTT_seconds'],
     ins_outside_clustered_specific_anls_df['VTT_seconds'],
     ins_outside_dispersed_specific_anls_df['VTT_seconds']],
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)
ax.set_title(f'VTT Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})')
ax.set_ylabel('VTT (seconds)')
plt.tight_layout()

plt.show()

### TKT

In [ ]:
ins_center_clustered_specific_anls_df

In [ ]:
''' Box plot '''
_ins_center_clustered_specific_anls_df = ins_center_clustered_specific_anls_df.copy()
_ins_center_clustered_specific_anls_df['TKT_tonkm'] = _ins_center_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_center_dispersed_specific_anls_df = ins_center_dispersed_specific_anls_df.copy()
_ins_center_dispersed_specific_anls_df['TKT_tonkm'] = _ins_center_dispersed_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_clustered_specific_anls_df = ins_outside_clustered_specific_anls_df.copy()
_ins_outside_clustered_specific_anls_df['TKT_tonkm'] = _ins_outside_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_dispersed_specific_anls_df = ins_outside_dispersed_specific_anls_df.copy()
_ins_outside_dispersed_specific_anls_df['TKT_tonkm'] = _ins_outside_dispersed_specific_anls_df['TKT_tonkm'] / 1000

fig, ax, bp = figure_plot.box_plot(
    data_list=[
        _ins_center_clustered_specific_anls_df,
        _ins_center_dispersed_specific_anls_df,
        _ins_outside_clustered_specific_anls_df,
        _ins_outside_dispersed_specific_anls_df
    ],
    col_name='TKT_tonkm',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    filename='boxplot_tkt_across_spatial_distributions_scenarios.png'
)

### Fleet size

In [ ]:
ins_center_clustered_specific_anls_df

In [ ]:
dispersed_shipment_metrics_df = agg_anls.aggregate_shipment_metrics_from_clean_folder(str(DISPERSED_OUTPUT_PATH), verbose=True)
dispersed_shipment_metrics_df

In [ ]:
ins_center_clustered_specific_anls_df = pd.merge(
    ins_center_clustered_specific_anls_df,
    shipment_metrics_df[['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'final_fleet_size']],
    left_on=['instance', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    right_on=['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    how='left'
)

ins_outside_clustered_specific_anls_df = pd.merge(
    ins_outside_clustered_specific_anls_df,
    shipment_metrics_df[['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'final_fleet_size']],
    left_on=['instance', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    right_on=['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    how='left'
)

ins_outside_dispersed_specific_anls_df = pd.merge(
    ins_outside_dispersed_specific_anls_df,
    dispersed_shipment_metrics_df[['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'final_fleet_size']],
    left_on=['instance', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    right_on=['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    how='left'
)

ins_center_dispersed_specific_anls_df = pd.merge(
    ins_center_dispersed_specific_anls_df,
    shipment_metrics_df[['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'final_fleet_size']],
    left_on=['instance', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    right_on=['instance_id', 'depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
    how='left'
)

In [ ]:
''' print the count of instances and the mean of each final fleet size for each scenario '''
ins_center_clustered_specific_anls_df.groupby('final_fleet_size').agg({'instance': 'count'})

In [ ]:
_concat_df = pd.concat([
    ins_center_clustered_specific_anls_df,
    ins_center_dispersed_specific_anls_df,
    ins_outside_clustered_specific_anls_df,
    ins_outside_dispersed_specific_anls_df
], ignore_index=True)



In [ ]:
data_dict = {
    'Center-Clustered': ins_center_clustered_specific_anls_df,
    'Center-Dispersed': ins_center_dispersed_specific_anls_df,
    'Outside-Clustered': ins_outside_clustered_specific_anls_df,
    'Outside-Dispersed': ins_outside_dispersed_specific_anls_df
}

fig, ax = figure_plot.nested_donut_chart(
    data_dict,
    group_col='final_fleet_size',
    # title='Fleet Size Distribution by Scenario',
    inner_colors=["#88ABA7", "#ddd2e9c4", "#4d8075", "#b7a4ca"],
    outer_color_list=["#dcbdc3", "#c18d97", "#844954CF", "#844954"],
    outer_cmap='coolwarm',
    show_inner_labels=False,
    show_inner_pct=False,
    show_outer_count=False,
    show_outer_labels=False,
    hole_radius=0.4,
    inner_label_size=5,
    dpi=350,
    outer_label_size=10,
    figsize=(5, 5),
    show_legend=False,
    figure_folder=OUTPUT_FIG_PATH,
    filename='nested_donut_fleet_size_by_scenario.png',
     transparent_bg=True
)

### Spatial index
Relations: (mean-dist-to-depot; clustering index; ) <-> (VKT, TT, TKT, Cost savings, scores, etc)

In [ ]:
_concat_df['class'] = _concat_df.apply(lambda row: f"{row['depot_location']}_{row['receiver_distribution']}", axis=1)
_concat_df

In [ ]:
ins_center_clustered_specific_anls_df['clustering_index_euclidean_km'].mean()

In [ ]:
ins_center_dispersed_specific_anls_df['clustering_index_euclidean_km'].mean()

In [ ]:

figure_plot.joint_scatter_plot(
    data_df=_concat_df,
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='clustering_index_euclidean_km',
    # size_group_col='total_cost_savings',
    size_group_col='collaboration_rate',
    color_group_col='class',
    show_size_legend=False,

)

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='clustering_index_euclidean_km',
    y_col='total_cost_savings',
    data_df=ins_center_dispersed_specific_anls_df,
    figure_size=(5.8,4),
    xlabel='Disperse index (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=False,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    # figure_folder=OUTPUT_FIG_PATH,
    # filename='scatter_regression_cost_savings_vs_clustering_index.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=_concat_df,
    figure_size=(5.8,4),
    xlabel='Mean receiver distance to depot (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=False,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    filename='scatter_regression_cost_savings_vs_mean_receiver_dist2Depot.png'
    ) 

## Penalty sensitivity

In [ ]:
all_metrics_af80_df = all_metrics_df[
    (all_metrics_df['allocation_factor'] == 0.8)
]



all_metric_af80_dispersed_df = all_metrics_af80_df[
    (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

all_metrics_af80_clustered_df = all_metrics_af80_df[
    (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

all_metric_af80_center_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.INSIDE.value)
]

all_metrics_af80_outside_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.OUTSIDE.value)
]

all_metrics_af80_center_dispersed_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.INSIDE.value
    ) & (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

all_metrics_af80_outside_dispersed_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.OUTSIDE.value
    ) & (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

all_metrics_af80_center_clustered_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.INSIDE.value
    ) & (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)  
]

all_metrics_af80_outside_clustered_df = all_metrics_af80_df[
    (all_metrics_af80_df['depot_location'] == DepotLocation.OUTSIDE.value
    ) & (all_metrics_af80_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)  
]


In [ ]:
all_metrics_af80_center_clustered_df.groupby('penalty')['collaboration_rate'].mean()

In [ ]:
all_metrics_af80_center_dispersed_df.query("penalty == 0.0056")

In [ ]:
all_metrics_af80_center_dispersed_df.query("penalty == 0.0098")

In [ ]:
all_metrics_af80_center_dispersed_df.query("penalty == 0.014")

In [ ]:
agg_metrics_center_random_df = agg_random_df_across_instances(all_metrics_center_random_df)
agg_metrics_outside_random_df = agg_random_df_across_instances(all_metrics_outside_random_df)

### collaboration rate

In [ ]:
boxplot_metric(all_metric_af80_dispersed_df,
               metric='collaboration_rate',
               title='Collaboration Rate Distribution (AF=0.8, Dispersed Receivers)',
)

boxplot_metric(all_metrics_af80_clustered_df,
                metric='collaboration_rate',
                title='Collaboration Rate Distribution (AF=0.8, Clustered Receivers)',
    )   

boxplot_metric(all_metric_af80_center_df,
                metric='collaboration_rate',
                title='Collaboration Rate Distribution (AF=0.8, Center Depot)',
    )

boxplot_metric(all_metrics_af80_outside_df,
                metric='collaboration_rate',
                title='Collaboration Rate Distribution (AF=0.8, Outside Depot)',
    )

In [ ]:
boxplot_metric(all_metrics_af80_center_dispersed_df ,
               metric='collaboration_rate',
               title='Collaboration Rate vs Penalty (Center Depot, dispersed Receivers)',  
               compare_iter0=False)
boxplot_metric(all_metrics_af80_outside_dispersed_df,
               metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Outside Depot, dispersed Receivers)',
                compare_iter0=False)
boxplot_metric(all_metrics_af80_center_clustered_df,
               metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Center Depot, Clustered Receivers)',
                compare_iter0=False)
boxplot_metric(all_metrics_af80_outside_clustered_df,
               metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Outside Depot, Clustered Receivers)',
                compare_iter0=False)


In [ ]:
# boxplot_metric(shrink_group_variance(all_metrics_center_random_df,
#                                      group_col='penalty',
#                                      value_col='collaboration_rate',
#                                      alpha=0.6,
#                                      center='median',
#                                      groups=[0.0003, 0.0008],
#                                      value_step=0.1),
#                metric='collaboration_rate',
#                title='Collaboration Rate vs Penalty (Center Depot, Random Receivers)',  
#                compare_iter0=False)
# boxplot_metric(shrink_group_variance(all_metrics_outside_random_df,
#                                      group_col='penalty',
#                                      value_col='collaboration_rate',
#                                      alpha=0.6,
#                                      center='median',
#                                      groups=[0.0003, 0.0008],
#                                      value_step=0.1),
#                metric='collaboration_rate',
#                 title='Collaboration Rate vs Penalty (Outside Depot, Random Receivers)',
#                 compare_iter0=False)

### VKT

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='VKT_km', 
               title='Total Freight VKT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='VKT_km', 
                title='Total Freight VKT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### VTT (travel time)

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='VTT_seconds', 
               title='Total Freight VTT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='VTT_seconds', 
                title='Total Freight VTT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Ton-km travelled (tkt)

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='TKT_tonkm', 
               title='Total Freight TKT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='TKT_tonkm', 
                title='Total Freight TKT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Fleet size

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='fleet_size', 
               title='Total Freight Fleet Size (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='fleet_size', 
                title='Total Freight Fleet Size (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Carrier score

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='final_carrier_score', 
               title='Carrier Score (Center Depot, Random Receivers)', 
               iter0_metric='iter0_carrier_score')
boxplot_metric(all_metrics_outside_random_df,
                metric='final_carrier_score', 
                title='Carrier Score (Outside Depot, Random Receivers)', 
                iter0_metric='iter0_carrier_score')

### Receiver score

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='total_receiver_scores', 
               title='Total Freight Receiver Scores (Center Depot, Random Receivers)', 
               compare_iter0=False)
boxplot_metric(all_metrics_outside_random_df,
                metric='total_receiver_scores', 
                title='Total Freight Receiver Scores (Outside Depot, Random Receivers)', 
                compare_iter0=False)

### extended TWs

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='total_extended_tw_hours', 
               title='Total Extended TW Hours (Center Depot, Random Receivers)', 
               compare_iter0=False)
boxplot_metric(all_metrics_outside_random_df,
                metric='total_extended_tw_hours', 
                title='Total Extended TW Hours (Outside Depot, Random Receivers)', 
                compare_iter0=False)

### Mean receiver distance to the depot
This should be set as the X , exporing the correlations with other variables (like collaboration rate, scores, etc.)

In [ ]:
scatter_3d(all_metrics_center_random_df,
           x_col='mean_receiver_dist_to_depot_euclidean_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='center-random-Mean receiver euclidean dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_center_random_df,
                      x_col='mean_receiver_dist_to_depot_euclidean_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='center-random-Mean receiver euclidean dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_center_random_df,
                   x_col='mean_receiver_dist_to_depot_euclidean_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high']
                   )

In [ ]:
scatter_3d(all_metrics_center_random_df,
           x_col='mean_receiver_dist_to_depot_network_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='center-random-Mean receiver nx dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_center_random_df,
                      x_col='mean_receiver_dist_to_depot_network_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='center-random-Mean receiver nx dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_center_random_df,
                   x_col='mean_receiver_dist_to_depot_network_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high'],
                    
                   )

In [ ]:
scatter_3d(all_metrics_outside_random_df,
           x_col='mean_receiver_dist_to_depot_network_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_outside_random_df,
                      x_col='mean_receiver_dist_to_depot_network_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_outside_random_df,
                   x_col='mean_receiver_dist_to_depot_network_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high'],
                    title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
                   )
basic_scatter_plot(all_metrics_outside_random_df,
                     x_col='mean_receiver_dist_to_depot_network_km',
                     y_col='collaboration_rate',
                    #  penalty_value=0.0056,
                     title='outside-random-Mean receiver nx dist to depot-collab rate at penalty=0.0056'
                     )

## Allocation sensitivity